<a href="https://colab.research.google.com/github/Prabhjot-Singh1/Deep-Learning-Lab/blob/main/END_SEM_PRACTICAL/DLL_practical_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# !pip install torch torchvision

import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms

In [2]:
BATCH_SIZE = 128
EPOCHS     = 10
LR         = 0.001
DEVICE     = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Using device: {DEVICE}")

Using device: cuda


In [4]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

train_set = torchvision.datasets.CIFAR10(root='./data', train=True,
                                          download=True, transform=transform)
test_set  = torchvision.datasets.CIFAR10(root='./data', train=False,
                                          download=True, transform=transform)

train_loader = torch.utils.data.DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True)
test_loader  = torch.utils.data.DataLoader(test_set,  batch_size=BATCH_SIZE, shuffle=False)

CLASSES = ('plane','car','bird','cat','deer','dog','frog','horse','ship','truck')
print("Data loaded!")

Data loaded!


In [5]:
# output = F(x) + x
# The block learns the residual F(x), not the full mapping
# Skip connection lets gradients flow back directly → fixes vanishing gradient

class ResidualBlock(nn.Module):
    def __init__(self, channels):
        super().__init__()

        self.conv1 = nn.Conv2d(channels, channels, kernel_size=3, padding=1, bias=False)
        self.bn1   = nn.BatchNorm2d(channels)
        self.relu  = nn.ReLU(inplace=True)

        self.conv2 = nn.Conv2d(channels, channels, kernel_size=3, padding=1, bias=False)
        self.bn2   = nn.BatchNorm2d(channels)

    def forward(self, x):
        identity = x                       # save input (skip connection)

        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))

        out = out + identity               # ADD skip connection
        out = self.relu(out)
        return out

In [6]:
# Conv(3→32) → 2x ResBlock(32) → Downsample(32→64) → 2x ResBlock(64) → GAP → FC

class MiniResNet(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()

        self.stem = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True)
        )

        self.stage1 = nn.Sequential(
            ResidualBlock(32),
            ResidualBlock(32)
        )

        self.downsample = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True)
        )

        self.stage2 = nn.Sequential(
            ResidualBlock(64),
            ResidualBlock(64)
        )

        self.gap = nn.AdaptiveAvgPool2d(1)
        self.fc  = nn.Linear(64, num_classes)

    def forward(self, x):
        x = self.stem(x)
        x = self.stage1(x)
        x = self.downsample(x)
        x = self.stage2(x)
        x = self.gap(x)
        x = x.view(x.size(0), -1)         # flatten
        x = self.fc(x)
        return x


model = MiniResNet().to(DEVICE)
print(model)
print(f"\nTotal parameters: {sum(p.numel() for p in model.parameters()):,}")

MiniResNet(
  (stem): Sequential(
    (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU(inplace=True)
  )
  (stage1): Sequential(
    (0): ResidualBlock(
      (conv1): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): ResidualBlock(
      (conv1): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(32, 32, kernel_size=(3, 3), stride=

In [7]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LR)

def train_one_epoch(epoch):
    model.train()
    total_loss, correct, total = 0, 0, 0

    for images, labels in train_loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)

        optimizer.zero_grad()
        outputs = model(images)
        loss    = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        _, predicted = outputs.max(1)
        correct += predicted.eq(labels).sum().item()
        total   += labels.size(0)

    acc = 100. * correct / total
    print(f"Epoch [{epoch}/{EPOCHS}]  Loss: {total_loss/len(train_loader):.4f}  Train Acc: {acc:.2f}%")


def evaluate():
    model.eval()
    correct, total = 0, 0

    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            outputs = model(images)
            _, predicted = outputs.max(1)
            correct += predicted.eq(labels).sum().item()
            total   += labels.size(0)

    acc = 100. * correct / total
    print(f"  → Test Accuracy: {acc:.2f}%\n")

In [8]:
for epoch in range(1, EPOCHS + 1):
    train_one_epoch(epoch)
    evaluate()

print("Training complete!")

Epoch [1/10]  Loss: 1.3703  Train Acc: 50.26%
  → Test Accuracy: 53.33%

Epoch [2/10]  Loss: 0.9699  Train Acc: 65.61%
  → Test Accuracy: 57.01%

Epoch [3/10]  Loss: 0.8103  Train Acc: 71.49%
  → Test Accuracy: 68.16%

Epoch [4/10]  Loss: 0.7041  Train Acc: 75.33%
  → Test Accuracy: 71.16%

Epoch [5/10]  Loss: 0.6174  Train Acc: 78.21%
  → Test Accuracy: 67.27%

Epoch [6/10]  Loss: 0.5548  Train Acc: 80.85%
  → Test Accuracy: 76.79%

Epoch [7/10]  Loss: 0.4984  Train Acc: 82.65%
  → Test Accuracy: 78.00%

Epoch [8/10]  Loss: 0.4573  Train Acc: 84.15%
  → Test Accuracy: 77.92%

Epoch [9/10]  Loss: 0.4232  Train Acc: 85.36%
  → Test Accuracy: 76.93%

Epoch [10/10]  Loss: 0.3846  Train Acc: 86.63%
  → Test Accuracy: 81.60%

Training complete!
